# A2 · The hospital simulator

**Goal:** build the hidden rule, generate 8 months of visits, and check the simulated hospital behaves like a real OPD *before* we trust it.

1. Write `expected_minutes` and `sample_noise` in [slotpred/world.py](../slotpred/world.py) (`TODO(you)`)
2. Run the tests until they pass
3. Generate the data
4. Sanity checks (given)
5. **The oracle** (`TODO(you)`): the best score any model could get

⚠️ We look only at the clean `world` here. The messy export (`raw`) is for A3, so don't explore it yet.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from slotpred.config import ROOT, TEST_START
from slotpred import world as w

## Step 2 - run the tests

Red means not right yet; read the failing test's name and message. Green means done. Rerun this cell after each edit to `world.py`.

In [ ]:
!"{sys.executable}" -m pytest "{ROOT / 'tests' / 'test_world.py'}" -q

## Step 3 - generate the data

In [ ]:
from slotpred import simulate as sim

world, raw = sim.main()
world.head()

## Step 4 - sanity checks

### 4a. Is it right-skewed, like real consults?

In [ ]:
y = world["actual_minutes"]
print(f"mean {y.mean():.1f}  median {y.median():.1f}  P80 {y.quantile(0.8):.1f}  max {y.max():.0f}")

plt.figure(figsize=(8, 3.5))
plt.hist(y, bins=100, range=(0, 60))
plt.axvline(y.median(), color="black", linestyle=":", label="median")
plt.axvline(y.mean(), color="black", linestyle="--", label="mean")
plt.xlabel("actual minutes"); plt.ylabel("visits"); plt.legend(); plt.show()

### 4b. Do departments and visit types come out in the right order?

Compare with the base table. The medians will differ a little: reports, interpreters, age and noise all shift them.

In [ ]:
world.pivot_table(index="department", columns="visit_type", values="actual_minutes", aggfunc="median").round(1)

### 4c. Can we see the drift?

Weekly median for Dermatology vs everyone else. Something should change on 1 July.

In [ ]:
weekly = (world.assign(week=pd.to_datetime(world["date"]).dt.to_period("W").dt.start_time,
                       group=np.where(world["department"] == "Dermatology", "Dermatology", "Others"))
              .groupby(["week", "group"])["actual_minutes"].median().unstack())

weekly.plot(figsize=(9, 3.5), marker=".")
plt.axvline(pd.Timestamp(TEST_START), color="red", linestyle="--", label="1 July")
plt.ylabel("weekly median minutes"); plt.legend(); plt.show()

## Step 5 - TODO(you): the oracle

The oracle knows the hidden rule exactly (`expected_minutes`), but not the noise. Its best predictions are:

- **P50** = expected × e^(−σ²/2)
- **P80** = expected × e^(−σ²/2 + 0.8416·σ)

On the **test period** (`date >= TEST_START`), compute:

| | Oracle | Fixed 15-min slot (today's rule) |
|---|---|---|
| MAE (P50 guess vs actual) | ? | use 15 as the guess |
| Coverage (actual ≤ P80) | ? | actual ≤ 15 |
| Average idle minutes (slot − actual, when positive) | ? | ? |

For the oracle's slot, use P80 **rounded up to 5 minutes** (`np.ceil(p80 / 5) * 5`). Use `w.NOISE_SIGMA` for σ.

In [ ]:
test = world[world["date"] >= TEST_START].copy()

# TODO(you): oracle P50, P80 and slot columns on `test`, then the metrics for both rows of the table.


## Questions - write your answers here

1. What's the oracle's MAE? This is the **irreducible error**: no model can beat it on average.

   *Answer:*

2. Is the oracle's coverage close to 80%? It isn't exactly 80% - why? (Hint: the slot is rounded up to 5 minutes.)

   *Answer:*

3. How does the fixed 15-minute slot compare on coverage and idle time? Which kind of visit does it hurt most?

   *Answer:*

4. (From the lesson) Why does σ = 0.28 make P80 about 27% bigger than the median, whatever the expected minutes are?

   *Answer:*